# Optimize Collections

Run the contribution optimizer and inspect the optimized collection schedule.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "reserve_study").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from reserve_study import ReserveOptimizer, ReserveStudy

study = ReserveStudy.from_directory("2026_brendan_plan")
study_result = study.run(projection_years=30)
scenario = study.scenario
optimizer = ReserveOptimizer(study_result)

analysis_year = scenario.assumptions.analysis_year
start_contribution = scenario.collection_schedule.annual_for_year(analysis_year).contribution
start_contribution

In [ ]:
result = optimizer.optimize(
    contribution_fn=optimizer.contribution_fn_three_linear_then_inflation,
    objective_fn=optimizer.objective_min_total_plus_smooth,
    initial_params=[start_contribution, 20000.0, 20000.0, 20000.0, 5, 5, 5],
    bounds=[
        (start_contribution, start_contribution),
        (0.0, 100000.0),
        (0.0, 100000.0),
        (0.0, 100000.0),
        (4, 6),
        (4, 6),
        (4, 6),
    ],
    start_year=analysis_year,
    projection_years=30,
    min_balance=200000.0,
    special_mode="zero",
    transform_fn=lambda values: optimizer.transform_contributions(values, round_to=100.0),
    method="differential_evolution",
    options={
        "maxiter": 25,
        "popsize": 8,
        "tol": 0.02,
        "mutation": (0.1, 1.0),
        "recombination": 0.7,
        "seed": 42,
        "polish": False,
        "disp": False,
    },
)
result.diagnostics

In [ ]:
result.assessments_df().head(10)

In [ ]:
result.projection_df().head(10)

In [ ]:
study_result.write_optimization_result(result)
study_result.scenario.paths.output_source_data_dir / "optimized_assessment_contributions.csv"